# Multi-Model GSS Polarization Analysis (Two Categories)

This notebook analyzes LLM polarization on GSS survey questions across multiple models.

**Two categories of GSS questions:**
- **Public Issues**: Policy/political questions → `default` template: *"Generate a statement by {name} on {topic}."*
- **Private Life**: Personal/lifestyle questions → `opinion` template: *"What would {name} say about {topic}?"*

**Models tested:** Llama, DeepSeek, Qwen, Gemma, Mistral, Phi

Results saved to `../results/` folder.

## 1. Setup and Configuration

In [1]:
import sys; sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import torch
import gc
import time
import glob
import re
from datetime import datetime
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("IMPORTS AND CONFIGURATION")
print("="*70)

print("\n[Importing local modules...]")

from config import NOMINATE_CSV, SYSTEM_MSG_POLITICIAN
print(f"  ✓ config.py")
print(f"      NOMINATE_CSV = {NOMINATE_CSV}")

from metrics_utils import compute_all_head_metrics, summarize_metrics, add_derived_metrics
print(f"  ✓ metrics_utils.py")

from prompt_utils import load_politicians, generate_politician_prompts, POLITICIAN_TEMPLATES
print(f"  ✓ prompt_utils.py")
print(f"      POLITICIAN_TEMPLATES: {list(POLITICIAN_TEMPLATES.keys())}")
for k, v in POLITICIAN_TEMPLATES.items():
    print(f"        - {k}: \"{v}\"")

print("\n[System info]")
print(f"  PyTorch: {torch.__version__}")
print(f"  CUDA: {torch.cuda.is_available()}, GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"    GPU {i}: {props.name} ({props.total_memory / 1e9:.1f} GB)")

/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/project/jevans/maxzhuyt/honest_llama_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


IMPORTS AND CONFIGURATION

[Importing local modules...]
  ✓ config.py
      NOMINATE_CSV = /project/jevans/maxzhuyt/gss_polarization/data/politicians.csv
  ✓ metrics_utils.py
  ✓ prompt_utils.py
      POLITICIAN_TEMPLATES: ['default', 'opinion', 'speech', 'tweet', 'interview']
        - default: "Generate a statement by {name} on {topic}."
        - opinion: "What would {name} say about {topic}?"
        - speech: "Write a brief speech excerpt from {name} discussing {topic}."
        - tweet: "Write a tweet from {name} about {topic}."
        - interview: "In an interview, how would {name} respond to a question about {topic}?"

[System info]
  PyTorch: 2.9.0+cu128
  CUDA: True, GPUs: 1
    GPU 0: NVIDIA A100-SXM4-40GB (42.3 GB)


In [2]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Model paths
MODELS_BASE = "/project/jevans/maxzhuyt/models"

# Models to analyze
MODELS = {
    "Llama-3.1-8B": f"{MODELS_BASE}/Meta-Llama-3.1-8B-Instruct",
    "DeepSeek-R1-Qwen-14B": f"{MODELS_BASE}/DeepSeek-R1-Distill-Qwen-14B",
    "Qwen3-8B": f"{MODELS_BASE}/Qwen3-8B",
    "Gemma-2-9b": f"{MODELS_BASE}/gemma-2-9b-it",
    "Mistral-7B-v0.2": f"{MODELS_BASE}/Mistral-7B-Instruct-v0.2",
    "Phi-4-reasoing-plus-7B": f"{MODELS_BASE}/Phi-4-reasoning-plus",
    #"Mistral-Small-24B": f"{MODELS_BASE}/Mistral-Small-24B-Instruct-2501",

}

# Output directory
OUTPUT_DIR = Path("llm_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Processing settings
BATCH_SIZE = 64
MAX_LENGTH = 128

# Category settings
CATEGORIES = {
    "public_issues": {
        "topics_csv": "public_issues.csv",
        "polarization_csv": "public_issues_polarization.csv",
        "template_name": "default",
    },
    "private_life": {
        "topics_csv": "private_life.csv",
        "polarization_csv": "private_life_polarization.csv",
        "template_name": "opinion",
    },
}

print("="*70)
print("CONFIGURATION")
print("="*70)
print(f"\n[Output]: {OUTPUT_DIR.absolute()}")
print(f"[Batch size]: {BATCH_SIZE}")
print(f"[Max length]: {MAX_LENGTH}")
print(f"\n[Models ({len(MODELS)}):")
for name, path in MODELS.items():
    print(f"  - {name}")
print(f"\n[Categories]:")
for cat, cfg in CATEGORIES.items():
    print(f"  {cat}: template='{cfg['template_name']}' → \"{POLITICIAN_TEMPLATES[cfg['template_name']]}\"")

CONFIGURATION

[Output]: /project/jevans/maxzhuyt/../results
[Batch size]: 64
[Max length]: 128

[Models (6):
  - Llama-3.1-8B
  - DeepSeek-R1-Qwen-14B
  - Qwen3-8B
  - Gemma-2-9b
  - Mistral-7B-v0.2
  - Phi-4-reasoing-plus-7B

[Categories]:
  public_issues: template='default' → "Generate a statement by {name} on {topic}."
  private_life: template='opinion' → "What would {name} say about {topic}?"


## 2. Load Topics from CSV Files

In [3]:
def load_topics_from_csv(csv_path: str) -> dict:
    df = pd.read_csv(csv_path)
    return dict(zip(df['Variable'], df['NaturalLanguageClause']))

def load_polarization_data(csv_path: str) -> pd.DataFrame:
    return pd.read_csv(csv_path)

# Topics to exclude from analysis (identified via bootstrap sensitivity analysis)
EXCLUDED_TOPICS = {
    "public_issues": ['hubbywk1', 'intlwhts', 'racdif1', 'racdif2', 'racdif3', 'racdif4', 'wlthwhts', 'workwhts'],
    "private_life": ['helpful', 'helpfulnv', 'helpfulv', 'marwht', 'reborn'],
}

print("="*70)
print("LOADING TOPICS AND POLARIZATION DATA")
print("="*70)

all_topics = {}
all_polarization = {}

for cat_name, cat_config in CATEGORIES.items():
    print(f"\n[{cat_name.upper()}]")
    
    topics = load_topics_from_csv(cat_config['topics_csv'])
    pol_df = load_polarization_data(cat_config['polarization_csv'])
    all_polarization[cat_name] = pol_df
    
    pol_vars = set(pol_df['variable'].tolist())
    topics_with_pol = {k: v for k, v in topics.items() if k in pol_vars}
    
    # Apply exclusions
    excluded = EXCLUDED_TOPICS.get(cat_name, [])
    all_topics[cat_name] = {k: v for k, v in topics_with_pol.items() if k not in excluded}
    
    print(f"  Topics CSV: {cat_config['topics_csv']} ({len(topics)} total)")
    print(f"  Polarization CSV: {cat_config['polarization_csv']} ({len(pol_df)} vars)")
    print(f"  Topics with polarization data: {len(topics_with_pol)}")
    print(f"  Excluded topics: {len(excluded)} - {excluded}")
    print(f"  Final topics to analyze: {len(all_topics[cat_name])}")
    print(f"  Template: '{cat_config['template_name']}'")

total = sum(len(t) for t in all_topics.values())
print(f"\nTOTAL: {total} topics across {len(CATEGORIES)} categories")

LOADING TOPICS AND POLARIZATION DATA

[PUBLIC_ISSUES]
  Topics CSV: public_issues.csv (172 total)
  Polarization CSV: public_issues_polarization.csv (134 vars)
  Topics with polarization data: 134
  Excluded topics: 8 - ['hubbywk1', 'intlwhts', 'racdif1', 'racdif2', 'racdif3', 'racdif4', 'wlthwhts', 'workwhts']
  Final topics to analyze: 126
  Template: 'default'

[PRIVATE_LIFE]
  Topics CSV: private_life.csv (98 total)
  Polarization CSV: private_life_polarization.csv (88 vars)
  Topics with polarization data: 76
  Excluded topics: 5 - ['helpful', 'helpfulnv', 'helpfulv', 'marwht', 'reborn']
  Final topics to analyze: 73
  Template: 'opinion'

TOTAL: 199 topics across 2 categories


## 3. Flexible Model Loading and Head Extraction

In [4]:
def detect_model_architecture(model) -> Dict[str, Any]:
    config = model.config
    
    if hasattr(config, 'head_dim') and config.head_dim is not None:
        head_dim = config.head_dim
    else:
        head_dim = config.hidden_size // config.num_attention_heads
    
    arch_info = {
        'num_layers': config.num_hidden_layers,
        'num_heads': config.num_attention_heads,
        'hidden_size': config.hidden_size,
        'head_dim': head_dim,
    }
    
    model_type = getattr(config, 'model_type', '').lower()
    
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        layers = model.model.layers
        if hasattr(layers[0], 'self_attn') and hasattr(layers[0].self_attn, 'o_proj'):
            arch_info['layers_path'] = 'model.model.layers'
            arch_info['attn_attr'] = 'self_attn'
            arch_info['o_proj_attr'] = 'o_proj'
            arch_info['architecture'] = f'{model_type}_llama_style'
            return arch_info
    
    raise ValueError(f"Could not detect architecture for model type: {model_type}")


def get_layers(model, arch_info):
    obj = model
    for attr in arch_info['layers_path'].split('.')[1:]:
        obj = getattr(obj, attr)
    return obj


def load_model_flexible(path: str, max_memory_per_gpu="39GiB"):
    print(f"  Loading from: {path}")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    
    tokenizer = AutoTokenizer.from_pretrained(path, use_fast=True, local_files_only=True)
    tokenizer.padding_side = 'left'
    tokenizer.truncation_side = 'left'
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    
    n_gpus = torch.cuda.device_count()
    max_memory = {i: max_memory_per_gpu for i in range(n_gpus)}
    
    model = AutoModelForCausalLM.from_pretrained(
        path, torch_dtype=dtype, device_map="auto",
        local_files_only=True, attn_implementation="eager",
        max_memory=max_memory
    )
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    
    arch_info = detect_model_architecture(model)
    print(f"  Architecture: {arch_info['architecture']}")
    print(f"  Layers: {arch_info['num_layers']}, Heads: {arch_info['num_heads']}, Head dim: {arch_info['head_dim']}")
    
    return model, tokenizer, arch_info


def unload_model(model, tokenizer):
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

In [5]:
@torch.no_grad()
def extract_heads_batched_flexible(
    model, tokenizer, arch_info,
    texts: List[str], system_msg: str,
    batch_size: int = 32, max_length: int = 128
) -> np.ndarray:
    model.eval()
    L, H, D_head = arch_info['num_layers'], arch_info['num_heads'], arch_info['head_dim']
    
    activations = []
    layer_outputs = [None] * L
    
    def get_hook(layer_idx):
        def hook(module, input, output):
            inp = input[0].detach()
            reshaped = inp.view(inp.shape[0], inp.shape[1], H, D_head)
            layer_outputs[layer_idx] = reshaped[:, -1, :, :].float().cpu().numpy()
        return hook
    
    hooks = []
    layers = get_layers(model, arch_info)
    for li in range(L):
        attn = getattr(layers[li], arch_info['attn_attr'])
        o_proj = getattr(attn, arch_info['o_proj_attr'])
        hooks.append(o_proj.register_forward_hook(get_hook(li)))
    
    try:
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            
            formatted_batch = []
            for t in batch:
                try:
                    formatted = tokenizer.apply_chat_template(
                        [{"role": "system", "content": system_msg}, {"role": "user", "content": t}],
                        tokenize=False, add_generation_prompt=True
                    )
                except:
                    try:
                        formatted = tokenizer.apply_chat_template(
                            [{"role": "user", "content": f"{system_msg}\n\n{t}"}],
                            tokenize=False, add_generation_prompt=True
                        )
                    except:
                        formatted = f"{system_msg}\n\nUser: {t}\n\nAssistant:"
                formatted_batch.append(formatted)
            
            enc = tokenizer(formatted_batch, return_tensors="pt", padding=True,
                           truncation=True, max_length=max_length).to(model.device)
            model(**enc)
            activations.append(np.stack(layer_outputs, axis=1))
    finally:
        for h in hooks:
            h.remove()
    
    return np.concatenate(activations, axis=0)

## 4. Analysis Pipeline

In [6]:
def run_topic_analysis(model, tokenizer, arch_info, topic_name, topic_desc, template):
    t0 = time.time()
    
    df_politicians = load_politicians(NOMINATE_CSV)
    prompts = generate_politician_prompts(topic_desc, df_politicians['fullname'].tolist(), template=template)
    labels = df_politicians['party_code'].values
    
    X_heads = extract_heads_batched_flexible(
        model, tokenizer, arch_info, prompts, SYSTEM_MSG_POLITICIAN,
        batch_size=BATCH_SIZE, max_length=MAX_LENGTH
    )
    
    metric_grids = compute_all_head_metrics(X_heads, labels, group_values=(100, 200))
    summary = summarize_metrics(metric_grids, topic_name)
    
    del X_heads, metric_grids
    print(f"      {topic_name}: Mahal={summary['Avg_Mahalanobis']:.4f} ({time.time()-t0:.1f}s)")
    return summary


def run_model_analysis(model_name, model_path, categories_topics, categories_config):
    print(f"\n{'='*70}")
    print(f"MODEL: {model_name}")
    print(f"{'='*70}")
    
    model, tokenizer, arch_info = load_model_flexible(model_path)
    all_results = []
    
    for cat_name, topics in categories_topics.items():
        template = POLITICIAN_TEMPLATES[categories_config[cat_name]['template_name']]
        print(f"\n  [{cat_name.upper()}] {len(topics)} topics, template: '{categories_config[cat_name]['template_name']}'")
        
        for idx, (topic_name, topic_desc) in enumerate(topics.items()):
            try:
                summary = run_topic_analysis(model, tokenizer, arch_info, topic_name, topic_desc, template)
                summary['category'] = cat_name
                summary['template'] = categories_config[cat_name]['template_name']
                summary['model'] = model_name
                all_results.append(summary)
                
                if (idx + 1) % 20 == 0:
                    gc.collect()
                    torch.cuda.empty_cache()
            except Exception as e:
                print(f"      ERROR {topic_name}: {e}")
    
    unload_model(model, tokenizer)
    
    df = pd.DataFrame(all_results)
    df = add_derived_metrics(df)
    return df

## 5. Run Analysis

In [9]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")


In [ ]:
# Run analysis for all models
model_results = {}

print("="*70)
print(f"STARTING ANALYSIS - {timestamp}")
print("="*70)

for model_name, model_path in MODELS.items():
    df_result = run_model_analysis(model_name, model_path, all_topics, CATEGORIES)
    model_results[model_name] = df_result
    
    # Save individual results
    out_path = OUTPUT_DIR / f"df_gss_2cat_{model_name}_{timestamp}.pkl"
    df_result.to_pickle(out_path)
    print(f"  Saved: {out_path}")

print(f"\n{'='*70}")
print("ALL MODELS COMPLETE")
print(f"{'='*70}")

STARTING ANALYSIS - 20260128_230153

MODEL: Llama-3.1-8B
  Loading from: /project/jevans/maxzhuyt/models/Meta-Llama-3.1-8B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  2.01s/it]


  Architecture: llama_llama_style
  Layers: 32, Heads: 32, Head dim: 128

  [PUBLIC_ISSUES] 126 topics, template: 'default'
  > Computing metrics for 1024 heads (Parallel)...
      abdefect: Mahal=3.7307 (9.3s)
  > Computing metrics for 1024 heads (Parallel)...
      abhlth: Mahal=3.8170 (6.4s)
  > Computing metrics for 1024 heads (Parallel)...
      abnomore: Mahal=3.9770 (6.6s)
  > Computing metrics for 1024 heads (Parallel)...
      abpoor: Mahal=3.9349 (6.1s)
  > Computing metrics for 1024 heads (Parallel)...
      abrape: Mahal=4.0525 (6.4s)
  > Computing metrics for 1024 heads (Parallel)...
      absingle: Mahal=4.1616 (6.5s)
  > Computing metrics for 1024 heads (Parallel)...
      abhelp1: Mahal=3.7132 (6.1s)
  > Computing metrics for 1024 heads (Parallel)...
      abhelp2: Mahal=4.3390 (5.7s)
  > Computing metrics for 1024 heads (Parallel)...
      abhelp3: Mahal=3.7555 (6.3s)
  > Computing metrics for 1024 heads (Parallel)...
      abhelp4: Mahal=4.1238 (6.4s)
  > Computing me

Loading checkpoint shards: 100%|██████████| 4/4 [01:38<00:00, 24.58s/it]


  Architecture: qwen2_llama_style
  Layers: 48, Heads: 40, Head dim: 128

  [PUBLIC_ISSUES] 126 topics, template: 'default'
  > Computing metrics for 1920 heads (Parallel)...
      abdefect: Mahal=2.6993 (10.6s)
  > Computing metrics for 1920 heads (Parallel)...
      abhlth: Mahal=2.7357 (7.9s)
  > Computing metrics for 1920 heads (Parallel)...
      abnomore: Mahal=2.6849 (8.2s)
  > Computing metrics for 1920 heads (Parallel)...
      abpoor: Mahal=2.6620 (7.7s)
  > Computing metrics for 1920 heads (Parallel)...
      abrape: Mahal=2.6996 (8.0s)
  > Computing metrics for 1920 heads (Parallel)...
      absingle: Mahal=2.7301 (7.8s)
  > Computing metrics for 1920 heads (Parallel)...
      abhelp1: Mahal=2.6725 (8.0s)
  > Computing metrics for 1920 heads (Parallel)...
      abhelp2: Mahal=2.6276 (7.3s)
  > Computing metrics for 1920 heads (Parallel)...
      abhelp3: Mahal=2.7025 (8.0s)
  > Computing metrics for 1920 heads (Parallel)...
      abhelp4: Mahal=2.6645 (7.0s)
  > Computing m

Loading checkpoint shards: 100%|██████████| 5/5 [00:30<00:00,  6.10s/it]


  Architecture: qwen3_llama_style
  Layers: 36, Heads: 32, Head dim: 128

  [PUBLIC_ISSUES] 126 topics, template: 'default'
  > Computing metrics for 1152 heads (Parallel)...
      abdefect: Mahal=2.7928 (6.5s)
  > Computing metrics for 1152 heads (Parallel)...
      abhlth: Mahal=2.8351 (5.6s)
  > Computing metrics for 1152 heads (Parallel)...
      abnomore: Mahal=2.8815 (5.6s)
  > Computing metrics for 1152 heads (Parallel)...
      abpoor: Mahal=2.7938 (5.6s)
  > Computing metrics for 1152 heads (Parallel)...
      abrape: Mahal=2.8525 (5.8s)
  > Computing metrics for 1152 heads (Parallel)...
      absingle: Mahal=2.9023 (5.2s)
  > Computing metrics for 1152 heads (Parallel)...
      abhelp1: Mahal=2.8434 (5.2s)
  > Computing metrics for 1152 heads (Parallel)...
      abhelp2: Mahal=2.8950 (5.1s)
  > Computing metrics for 1152 heads (Parallel)...
      abhelp3: Mahal=2.8549 (5.3s)
  > Computing metrics for 1152 heads (Parallel)...
      abhelp4: Mahal=2.8242 (5.1s)
  > Computing me

## 5b. Load Previously Saved Results (Optional)

In [6]:
# Load all saved results from ../results/
result_files = list(OUTPUT_DIR.glob("df_gss_2cat_*.pkl"))
print(f"Found {len(result_files)} result files:")
for f in sorted(result_files):
    print(f"  - {f.name}")

# Parse and load latest version of each model
file_info = {}
for f in result_files:
    match = re.match(r"df_gss_2cat_(.+?)_(\d{8}_\d{6})\.pkl$", f.name)
    if match:
        model_name, ts = match.groups()
        if model_name not in file_info or ts > file_info[model_name][1]:
            file_info[model_name] = (f, ts)

model_results = {}
for model_name, (filepath, ts) in file_info.items():
    df = pd.read_pickle(filepath)
    model_results[model_name] = df
    print(f"Loaded {model_name}: {len(df)} topics")

print(f"\nTotal models loaded: {len(model_results)}")

Found 7 result files:
  - df_gss_2cat_DeepSeek-R1-Qwen-14B_20260128_230153.pkl
  - df_gss_2cat_Gemma-2-9b_20260128_230153.pkl
  - df_gss_2cat_Llama-3.1-8B_20260128_230153.pkl
  - df_gss_2cat_Mistral-7B-v0.2_20260128_230153.pkl
  - df_gss_2cat_Phi-4-reasoing-plus-7B_20260128_230153.pkl
  - df_gss_2cat_Qwen3-8B_20260128_230153.pkl
  - df_gss_2cat_combined_20260128_230153.pkl
Loaded Llama-3.1-8B: 199 topics
Loaded combined: 1194 topics
Loaded DeepSeek-R1-Qwen-14B: 199 topics
Loaded Mistral-7B-v0.2: 199 topics
Loaded Gemma-2-9b: 199 topics
Loaded Qwen3-8B: 199 topics
Loaded Phi-4-reasoing-plus-7B: 199 topics

Total models loaded: 7


## 6. Compute Correlations

In [7]:
def compute_correlations(df_llm, df_gss, model_name, category):
    df_merged = df_llm[['Topic', 'Avg_Mahalanobis']].merge(
        df_gss[['variable', 'polarization']].rename(
            columns={'variable': 'Topic', 'polarization': 'GSS_Polarization'}
        ), on='Topic', how='inner'
    )
    return {
        'model': model_name,
        'category': category,
        'n_topics': len(df_merged),
        'pearson': df_merged['Avg_Mahalanobis'].corr(df_merged['GSS_Polarization'], method='pearson'),
        'spearman': df_merged['Avg_Mahalanobis'].corr(df_merged['GSS_Polarization'], method='spearman'),
    }

print("="*70)
print("CORRELATION ANALYSIS")
print("="*70)

correlation_results = []

for model_name, df_model in model_results.items():
    print(f"\n[{model_name}]")
    for cat_name in CATEGORIES.keys():
        df_cat = df_model[df_model['category'] == cat_name]
        if len(df_cat) == 0:
            continue
        
        corr = compute_correlations(df_cat, all_polarization[cat_name], model_name, cat_name)
        correlation_results.append(corr)
        print(f"  {cat_name}: r={corr['pearson']:.3f}, ρ={corr['spearman']:.3f} (n={corr['n_topics']})")

df_correlations = pd.DataFrame(correlation_results)
print("\n" + "="*70)
print("CORRELATION SUMMARY")
print("="*70)
display(df_correlations.sort_values(['category', 'pearson'], ascending=[True, False]))

CORRELATION ANALYSIS

[Llama-3.1-8B]
  public_issues: r=0.581, ρ=0.555 (n=126)
  private_life: r=0.532, ρ=0.447 (n=73)

[combined]
  public_issues: r=0.075, ρ=0.167 (n=756)
  private_life: r=0.046, ρ=0.107 (n=438)

[DeepSeek-R1-Qwen-14B]
  public_issues: r=0.437, ρ=0.403 (n=126)
  private_life: r=0.427, ρ=0.425 (n=73)

[Mistral-7B-v0.2]
  public_issues: r=0.709, ρ=0.687 (n=126)
  private_life: r=0.637, ρ=0.578 (n=73)

[Gemma-2-9b]
  public_issues: r=0.642, ρ=0.633 (n=126)
  private_life: r=0.578, ρ=0.512 (n=73)

[Qwen3-8B]
  public_issues: r=0.666, ρ=0.639 (n=126)
  private_life: r=0.476, ρ=0.421 (n=73)

[Phi-4-reasoing-plus-7B]
  public_issues: r=0.535, ρ=0.484 (n=126)
  private_life: r=0.598, ρ=0.576 (n=73)

CORRELATION SUMMARY


,model,category,n_topics,pearson,spearman
7,Mistral-7B-v0.2,private_life,73,0.637335,0.577912
13,Phi-4-reasoing-plus-7B,private_life,73,0.598264,0.576123
9,Gemma-2-9b,private_life,73,0.577838,0.512442
1,Llama-3.1-8B,private_life,73,0.531842,0.446570
11,Qwen3-8B,private_life,73,0.476271,0.421270
5,DeepSeek-R1-Qwen-14B,private_life,73,0.427157,0.424695
3,combined,private_life,438,0.045704,0.106758
6,Mistral-7B-v0.2,public_issues,126,0.708510,0.686683
10,Qwen3-8B,public_issues,126,0.665765,0.638959
8,Gemma-2-9b,public_issues,126,0.642302,0.633200


## 7. Visualization

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use a professional seaborn theme
sns.set_theme(style='whitegrid', context='talk', palette='deep')

# Create comparison bar plot by category
fig, axes = plt.subplots(1, len(CATEGORIES), figsize=(14, 6))

for idx, cat_name in enumerate(CATEGORIES.keys()):
    ax = axes[idx] if hasattr(axes, '__len__') else axes
    df_cat = df_correlations[df_correlations['category'] == cat_name].sort_values('pearson', ascending=True)
    
    y = np.arange(len(df_cat))
    width = 0.35
    
    ax.barh(y - width/2, df_cat['pearson'], width, label='Pearson r', alpha=0.9)
    ax.barh(y + width/2, df_cat['spearman'], width, label='Spearman ρ', alpha=0.9)
    
    ax.set_yticks(y)
    ax.set_yticklabels(df_cat['model'])
    ax.set_xlabel('Correlation with GSS Survey')
    
    # Use actual template text (not just the template key)
    template_key = CATEGORIES[cat_name]['template_name']
    template_text = POLITICIAN_TEMPLATES.get(template_key, template_key)
    ax.set_title(f"{cat_name.replace('_', ' ').title()}\n(template: {template_text})")
    
    ax.legend(frameon=False)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.6)

plt.tight_layout()
fig_path = OUTPUT_DIR / f"multimodel_2cat_correlation_{timestamp}.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight', facecolor='white')
print(f"Saved: {fig_path}")
plt.show()

NameError: name 'CATEGORIES' is not defined

## 8. Save Combined Results

In [ ]:
# Combine all results
df_combined = pd.concat(model_results.values(), ignore_index=True)

# Save (removed the word 'combined' from filenames)
combined_pkl = OUTPUT_DIR / f"df_gss_2cat_{timestamp}.pkl"
combined_csv = OUTPUT_DIR / f"df_gss_2cat_{timestamp}.csv"
corr_csv = OUTPUT_DIR / f"multimodel_2cat_correlations_{timestamp}.csv"

df_combined.to_pickle(combined_pkl)
df_combined.to_csv(combined_csv, index=False)
df_correlations.to_csv(corr_csv, index=False)

print("="*70)
print("FILES SAVED")
print("="*70)
print(f"\nOutput directory: {OUTPUT_DIR.absolute()}")
print(f"\nFiles:")
print(f"  - {combined_pkl.name}")
print(f"  - {combined_csv.name}")
print(f"  - {corr_csv.name}")
print(f"  - multimodel_2cat_correlation_{timestamp}.png")

## 9. Summary

In [ ]:
print("="*70)
print("ANALYSIS SUMMARY")
print("="*70)

print(f"\n[Run Info]")
print(f"  Timestamp: {timestamp}")
print(f"  Output: {OUTPUT_DIR.absolute()}")

print(f"\n[Topics]")
for cat_name, topics in all_topics.items():
    print(f"  {cat_name}: {len(topics)} topics")

print(f"\n[Models Analyzed]: {len(model_results)}")
for name in model_results.keys():
    print(f"  - {name}")

print(f"\n[Correlations by Category]")
for cat_name in CATEGORIES.keys():
    print(f"\n  {cat_name.upper()} (template: {CATEGORIES[cat_name]['template_name']})")
    df_cat = df_correlations[df_correlations['category'] == cat_name].sort_values('pearson', ascending=False)
    for _, row in df_cat.iterrows():
        print(f"    {row['model']:30s}: r={row['pearson']:.3f}, ρ={row['spearman']:.3f}")

print(f"\n[Best Models by Category]")
for cat_name in CATEGORIES.keys():
    df_cat = df_correlations[df_correlations['category'] == cat_name]
    best = df_cat.loc[df_cat['pearson'].idxmax()]
    print(f"  {cat_name}: {best['model']} (r={best['pearson']:.3f})")